# Exploration of Survey Property Maps — DP2 (USDF) — Native HEALPix Representation

This notebook is derived from `2026-03-11_ExploreDP2_SurveyPropertyMaps.ipynb`.

It adopts an **alternative visualisation approach** inspired by `read_healsparse.ipynb`:
instead of displaying pipeline-generated PNG images or using `skyproj`,
each `HealSparseMap` is converted to a dense HEALPix map (`healsparse → healpy`)
and displayed with `hp.mollview` / `hp.gnomview`, which provides:

- fine control over the projection (Mollweide, gnomonic, orthographic…),
- a coordinate grid (graticule) with coordinate labels,
- a Galactic plane overlay,
- adaptive colour normalisation (linear, LogNorm, SymLogNorm).

**Table of Contents**

1. Imports
2. Butler configuration
3. Utility function definitions
4. Butler initialisation
5. Explore available collections in `dp2_prep`
6. Search for Survey Property Map dataset types
7. Separation: HealSparseMap objects vs PNG images
8. Availability of HealSparse maps in the target collection
9A. Display configuration flags
9B. Helper function — Galactic plane
9C. HEALPix display — 2×3 band grid (healpy)
9D. PNG fallback — 2×3 band grid
10. Map value at the reference field
11. Statistics over a region around the reference field
12. Local visualisation (pcolormesh)
13. All-sky healpy view — loop over all available maps
14. Notes and next steps

| Field | Value |
|---|---|
| **Author** | Sylvie Dagoret-Campagne |
| **Creation date** | 2026-03-12 |
| **Environment** | USDF RSP — `LSST` kernel (`lsst_distrib` stack) |
| **DP2 reference notebook** | `2026-03-11_ExploreDP2_SurveyPropertyMaps.ipynb` |
| **HealSparse notebook** | `read_healsparse.ipynb` |
| **DP2 DRP Campaigns** | https://rubinobs.atlassian.net/wiki/spaces/DM/pages/661192727/LSSTCam+Intermittent+DRP+Runs |

| Dataset type | Description | Unit |
|---|---|---|
| `deepCoadd_exposure_time_consolidated_map_sum` | Total exposure time accumulated per sky position | **second** |
| `deepCoadd_epoch_consolidated_map_min` | Epoch of the earliest observation | **MJD** |
| `deepCoadd_epoch_consolidated_map_max` | Epoch of the latest observation | **MJD** |
| `deepCoadd_epoch_consolidated_map_mean` | Mean epoch of all contributing observations | **MJD** |
| `deepCoadd_psf_size_consolidated_map_weighted_mean` | Weighted mean of the PSF characteristic width (determinant radius) | **pixel** |
| `deepCoadd_psf_e1_consolidated_map_weighted_mean` | Weighted mean of PSF ellipticity component e1 | dimensionless |
| `deepCoadd_psf_e2_consolidated_map_weighted_mean` | Weighted mean of PSF ellipticity component e2 | dimensionless |
| `deepCoadd_psf_maglim_consolidated_map_weighted_mean` | Weighted mean of the PSF 5σ magnitude limit | **mag AB** |
| `deepCoadd_sky_background_consolidated_map_weighted_mean` | Weighted mean of the sky background level | **nJy** |
| `deepCoadd_sky_noise_consolidated_map_weighted_mean` | Weighted mean of the sky background standard deviation | **nJy** |

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.lines as mlines
from matplotlib.colors import LogNorm, SymLogNorm, Normalize
import pandas as pd
import os
import re
from pathlib import Path
from IPython.display import display

# LSST middleware
from lsst.daf.butler import Butler

# Astropy — Galactic coordinates for the Galactic plane overlay
from astropy.coordinates import SkyCoord, Galactic
import astropy.units as u

# healpy — native HEALPix visualisation (Mollweide, gnomview, etc.)
import healpy as hp

# healsparse — read HealSparseMap objects, convert to dense HEALPix
try:
    import healsparse
    HAS_HEALSPARSE = True
    print('healsparse available')
except ImportError:
    HAS_HEALSPARSE = False
    print('healsparse not available — HealSparse sections will be skipped')

# skyproj (optional — kept for compatibility)
try:
    import skyproj
    HAS_SKYPROJ = True
    print('skyproj available')
except ImportError:
    HAS_SKYPROJ = False
    print('skyproj not available')

print('All imports done.')

## 2. Butler configuration

In [ ]:
# ── Butler repository at USDF ─────────────────────────────────────────────────
REPO = 'dp2_prep'

# ── Target collection for survey property maps ────────────────────────────────
COLLECTION = 'LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545'

# ── Skymap ────────────────────────────────────────────────────────────────────
SKYMAP_NAME = 'lsst_cells_v2'
INSTRUMENT  = 'LSSTCam'

# ── Photometric bands ─────────────────────────────────────────────────────────
BANDS    = ['u', 'g', 'r', 'i', 'z', 'y']
BAND_REF = 'r'
band_order = {b: i for i, b in enumerate(BANDS)}

# ── Reference field coordinates ───────────────────────────────────────────────
RA_ECDFS  = 53.13
DEC_ECDFS = -28.10

RA_COSMOS  = 150.12
DEC_COSMOS = 2.21

FIELD   = 'COSMOS'
RA_CEN  = RA_COSMOS
DEC_CEN = DEC_COSMOS

# ── HEALPix resolution for HealSparse → healpy conversion ────────────────────
# nside=32  : fast all-sky view (~1.8°/pixel)
# nside=64  : good resolution/memory trade-off (~0.9°/pixel)
# nside=128 : high resolution (~0.46°/pixel) — slower
NSIDE_DISPLAY = 64

# ── Output directories ────────────────────────────────────────────────────────
NB_TAG   = 'DP2_SPMAP_HEALPIX'
DIR_DATA = f'data_{NB_TAG}'
DIR_FIGS = f'figs_{NB_TAG}'
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f'Data : {os.path.abspath(DIR_DATA)}')
print(f'Figs : {os.path.abspath(DIR_FIGS)}')

## 3. Utility function definitions

### 3A. HealSparseMap → dense HEALPix map conversion

The function `hspmap_to_healpix` converts a `HealSparseMap` object to a standard
HEALPix array (RING ordering) via `hsp.generate_healpix_map(nside)`,
then replaces out-of-coverage pixels (sentinel value) with `hp.UNSEEN`
so that `healpy` masks them correctly.

In [ ]:
def hspmap_to_healpix(hsp, nside=NSIDE_DISPLAY):
    """
    Convert a HealSparseMap to a dense HEALPix array (RING ordering, given nside).

    Parameters
    ----------
    hsp : healsparse.HealSparseMap
    nside : int
        Resolution of the output HEALPix map.

    Returns
    -------
    hpmap : np.ndarray, shape=(12*nside**2,)
        HEALPix map with hp.UNSEEN at out-of-coverage pixels.
    """
    hpmap = hsp.generate_healpix_map(nside=nside)
    # Replace the sentinel value with hp.UNSEEN
    sentinel = hsp.sentinel
    hpmap[hpmap == sentinel] = hp.UNSEEN
    return hpmap


print('hspmap_to_healpix defined.')

### 3B. Adaptive colour normalisation

Identical to the reference notebook version, but reused here for
`healpy` displays (vmin/vmax passed to `hp.mollview`).

In [ ]:
# Maps with signed values (require SymLogNorm)
SYMLOG_MAPS = {
    'deepCoadd_dcr_ddec_consolidated_map_weighted_mean',
    'deepCoadd_dcr_dra_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e1_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e2_consolidated_map_weighted_mean',
    'deepCoadd_psf_e1_consolidated_map_weighted_mean',
    'deepCoadd_psf_e2_consolidated_map_weighted_mean',
}


def compute_vmin_vmax(hpmap, p_low=1, p_high=99):
    """
    Compute the p_low and p_high percentiles on valid pixels (≠ hp.UNSEEN).

    Returns
    -------
    vmin, vmax : float
    """
    valid = hpmap[hpmap != hp.UNSEEN]
    valid = valid[np.isfinite(valid)]
    if valid.size == 0:
        return None, None
    return float(np.percentile(valid, p_low)), float(np.percentile(valid, p_high))


print('compute_vmin_vmax defined.')

### 3C. Galactic plane overlay on a healpy view

The projection used by healpy (Mollweide, gnomview…) does not accept standard
Matplotlib drawing commands directly: one must use `hp.projplot`, which
projects (RA, Dec) coordinates onto the figure plane.

In [ ]:
def galactic_plane_radec(n_points=2000):
    """
    Return (ra, dec) arrays tracing the Galactic equator (b=0) in ICRS.
    """
    gal_lon = np.linspace(0, 360, n_points)
    gal_lat = np.zeros(n_points)
    coords  = SkyCoord(
        l=gal_lon * u.deg,
        b=gal_lat * u.deg,
        frame=Galactic
    ).icrs
    return coords.ra.deg, coords.dec.deg


GP_RA, GP_DEC = galactic_plane_radec(n_points=2000)
print(f'Galactic plane: {len(GP_RA)} sample points computed.')


def overlay_galactic_plane(color='red', lw=1.2, alpha=0.8, label='Galactic plane'):
    """
    Overlay the Galactic plane on the current healpy figure.

    Must be called **after** hp.mollview / hp.gnomview (same active figure).
    Uses hp.projplot to project (RA, Dec) coordinates onto the figure plane.
    """
    # healpy expects (theta, phi) colatitude/longitude coordinates,
    # but with lonlat=True one can pass (lon, lat) in degrees.
    # We pass (RA, Dec) with coord='C' (equatorial) -> healpy projects them.
    hp.projplot(
        GP_RA, GP_DEC,
        lonlat=True,
        coord=['C'],
        color=color,
        lw=lw,
        alpha=alpha,
        label=label
    )


print('overlay_galactic_plane defined.')

### 3D. Main HEALPix display function — `show_hpmap_healpy`

Displays a HealSparse map using `hp.mollview` for the Mollweide projection
(same approach as `read_healsparse.ipynb`), with:
- a graticule (coordinate grid),
- latitude and longitude labels,
- an optional Galactic plane overlay.

In [ ]:
def show_hpmap_healpy(
    hsp,
    title='',
    unit='',
    nside=NSIDE_DISPLAY,
    cmap='viridis',
    norm_mode='hist',
    use_log=False,
    linthresh=1.0,
    map_name='',
    show_gp=True,
    lon_0=0,
    figsize=(12, 6),
):
    """
    Display a HealSparseMap using hp.mollview.

    Parameters
    ----------
    hsp : healsparse.HealSparseMap
    title : str
        Figure title.
    unit : str
        Colour bar label.
    nside : int
        HEALPix resolution for the conversion.
    cmap : str
        Matplotlib colormap.
    norm_mode : str
        'hist'   -> histogram normalisation (healpy built-in),
        'log'    -> logarithmic scale (vmin/vmax from percentiles),
        'symlog' -> SymLogNorm,
        'linear' -> linear (vmin/vmax from percentiles).
    use_log : bool
        Shortcut: if True, selects 'log' or 'symlog' based on map_name.
    linthresh : float
        Linear threshold for SymLogNorm.
    map_name : str
        Dataset type name (used to choose log/symlog automatically).
    show_gp : bool
        If True, overlay the Galactic plane.
    lon_0 : float
        Central longitude for the Mollweide projection (degrees).
    figsize : tuple
        Figure size.
    """
    hpmap = hspmap_to_healpix(hsp, nside=nside)
    vmin, vmax = compute_vmin_vmax(hpmap)

    # Choose the normalisation mode
    if use_log:
        norm_mode = 'symlog' if (map_name in SYMLOG_MAPS) else 'log'

    if norm_mode == 'hist':
        norm_kw = dict(norm='hist')
    elif norm_mode == 'log' and vmin is not None and vmin > 0:
        hpmap = np.log10(np.where(hpmap > 0, hpmap, np.nan))
        hpmap[~np.isfinite(hpmap)] = hp.UNSEEN
        vmin2, vmax2 = compute_vmin_vmax(hpmap)
        norm_kw = dict(min=vmin2, max=vmax2)
        unit = f'log10({unit})' if unit else 'log10(value)'
    elif norm_mode == 'symlog':
        norm_kw = dict(min=vmin, max=vmax)
    else:  # linear
        norm_kw = dict(min=vmin, max=vmax)

    plt.figure(figsize=figsize)
    hp.mollview(
        hpmap,
        title=title,
        cmap=cmap,
        unit=unit,
        rot=(lon_0, 0, 0),
        coord=['C'],
        badcolor='lightgrey',   # colour of hp.UNSEEN / NaN pixels
        bgcolor='white',        # background colour outside the ellipse
        hold=False,
        **norm_kw
    )

    # Graticule
    hp.graticule(dpar=30, dmer=30, color='white', alpha=0.5, lw=0.5)

    # Latitude labels
    for lat in [-60, -30, 0, 30, 60]:
        hp.projtext(
            lon_0 + 2, lat,
            f'{lat:+d}°',
            lonlat=True, coord='C',
            color='black', fontsize=8
        )

    # Longitude labels
    for lon in range(0, 360, 60):
        hp.projtext(
            lon, 0,
            f'{lon}°',
            lonlat=True, coord='C',
            color='black', fontsize=8
        )

    # Galactic plane
    if show_gp:
        overlay_galactic_plane(color='red', lw=1.2, alpha=0.8)

    plt.tight_layout()
    plt.show()


print('show_hpmap_healpy defined.')

### 3E. 2×3 band grid display — `display_hpmap_6bands_healpy`

Produces a 6-panel figure (one per band) using `hp.mollview`
with a common normalisation computed from the first available band.

In [ ]:
def display_hpmap_6bands_healpy(
    map_name,
    butler_obj,
    collection=None,
    bands=BANDS,
    skymap=SKYMAP_NAME,
    nside=NSIDE_DISPLAY,
    use_log=True,
    linthresh=1.0,
    cmap='viridis',
    figsize=(20, 10),
):
    """
    Display the `map_name` map for all 6 bands in a 2x3 grid
    using hp.mollview (native healpy approach).

    The colour normalisation is derived from the 1st-99th percentile of the
    first available band, then applied to all bands.
    """
    if collection is None:
        collection = COLLECTION

    short = (
        map_name
        .replace('deepCoadd_', '')
        .replace('_consolidated', '')
    )
    is_symlog = map_name in SYMLOG_MAPS

    # ── Retrieve maps ─────────────────────────────────────────────────────────
    hpmaps = {}
    for band in bands:
        try:
            hsp = butler_obj.get(
                map_name,
                dataId={'band': band, 'skymap': skymap},
                collections=collection
            )
            hpmaps[band] = hspmap_to_healpix(hsp, nside=nside)
        except Exception as e:
            print(f'  [{band}] not available: {type(e).__name__}')

    if not hpmaps:
        print(f'No map available for {map_name}.')
        return

    # ── Common normalisation ──────────────────────────────────────────────────
    first_hpmap = next(iter(hpmaps.values()))
    vmin, vmax = compute_vmin_vmax(first_hpmap)

    if use_log and vmin is not None:
        if is_symlog:
            norm_kw = dict(min=vmin, max=vmax)
        elif vmin > 0:
            norm_kw = dict(norm='hist')
        else:
            norm_kw = dict(min=vmin, max=vmax)
    else:
        norm_kw = dict(min=vmin, max=vmax)

    # ── 2x3 figure ────────────────────────────────────────────────────────────
    nrows, ncols = 2, 3
    fig = plt.figure(figsize=figsize)
    fig.suptitle(
        f'Survey Property Map — {short}\n{collection}',
        fontsize=13, fontweight='bold'
    )

    for idx, band in enumerate(bands):
        if band not in hpmaps:
            continue

        # hp.mollview draws into the current figure via sub=(nrows, ncols, idx+1)
        hp.mollview(
            hpmaps[band],
            title=f'band = {band}',
            cmap=cmap,
            coord=['C'],
            sub=(nrows, ncols, idx + 1),
            badcolor='lightgrey',   # colour of hp.UNSEEN / NaN pixels
            bgcolor='white',        # background colour outside the ellipse
            **norm_kw
        )
        hp.graticule(dpar=30, dmer=60, color='white', alpha=0.4, lw=0.4)

        # Galactic plane
        overlay_galactic_plane(color='red', lw=0.8, alpha=0.6)

    plt.show()


print('display_hpmap_6bands_healpy defined.')

### 3F. PNG fallback function — `display_plots_6bands`

In [ ]:
def get_plot_path(dataset_type, band, skymap=SKYMAP_NAME, collection=COLLECTION):
    try:
        uri = butler.getURI(
            dataset_type,
            dataId={'band': band, 'skymap': skymap},
            collections=collection
        )
        return uri.path
    except Exception as e:
        print(f'  Not found ({band}): {type(e).__name__}: {e}')
        return None


def display_plots_6bands(plot_name, df_plots, bands=BANDS, figsize=(20, 12)):
    """
    Display SurveyWidePropertyMapPlot PNG images for all 6 bands
    in a 2x3 grid.
    """
    df_sub = df_plots[(df_plots['dataset_type'] == plot_name) & df_plots['ok']]
    if df_sub.empty:
        print(f'No PNG images found for: {plot_name}')
        return

    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, layout='constrained')
    axes_flat = axes.flatten()

    short = (
        plot_name
        .replace('propertyMapSurvey_deepCoadd_', '')
        .replace('_SurveyWidePropertyMapPlot', '')
    )

    for idx, band in enumerate(bands):
        ax  = axes_flat[idx]
        row = df_sub[df_sub['band'] == band]
        if row.empty:
            ax.set_visible(False)
            continue
        img = mpimg.imread(row.iloc[0]['path'])
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'band = {band}', fontsize=12, fontweight='bold')

    for idx in range(len(bands), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.suptitle(
        f'Survey Property Map Plots — {short}\n'
        f'(PNG images — fallback)\n{COLLECTION}',
        fontsize=12, fontweight='bold'
    )
    plt.show()


print('PNG fallback functions defined.')

## 4. Butler initialisation

In [ ]:
butler   = Butler(REPO)
registry = butler.registry
print(f'Butler instantiated on repository: {REPO}')

In [ ]:
try:
    skymap_obj = butler.get('skyMap', skymap=SKYMAP_NAME, collections=COLLECTION)
    print(f'Skymap {SKYMAP_NAME} loaded: {type(skymap_obj)}')
except Exception as e:
    print(f'Skymap not available: {type(e).__name__}: {e}')

## 5. Explore available collections in `dp2_prep`

In [ ]:
all_collections = sorted(registry.queryCollections())
print(f'Total number of collections: {len(all_collections)}')

In [ ]:
keywords = ['survey', 'prop', 'map', 'consolidated', 'spm', 'dp2-prep']
spm_candidates = [
    c for c in all_collections
    if any(kw.lower() in c.lower() for kw in keywords)
]
print(f'Candidate collections ({len(spm_candidates)}):')
for c in spm_candidates:
    print(' ', c)

## 6. Search for Survey Property Map dataset types

In [ ]:
patterns = [
    '*consolidated_map*',
    '*survey_property*',
    '*property_map*',
    '*healsparse*',
    '*healSparse*',
]

found_types = set()
for pattern in patterns:
    try:
        for dtype in registry.queryDatasetTypes(expression=pattern):
            found_types.add(dtype.name)
    except Exception as e:
        print(f"Pattern '{pattern}': {e}")

print(f'\nCandidate dataset types found ({len(found_types)}):')
for name in sorted(found_types):
    print(' ', name)

## 7. Separation: HealSparseMap objects vs PNG images

In [ ]:
type_spmap  = []
type_spplot = []
for ftype in found_types:
    if 'PropertyMapPlot' in ftype:
        type_spplot.append(ftype)
    elif 'deepCoadd_' in ftype and '_map_' in ftype:
        type_spmap.append(ftype)

print(f'{len(type_spmap)} HealSparseMap types, {len(type_spplot)} PNG plot types.')
print('\nHealSparseMap:')
for t in sorted(type_spmap):
    print(' ', t)

## 8. Availability of HealSparse maps in the target collection

In [ ]:
HPMAP_DATASET_TYPES = [
    'deepCoadd_dcr_ddec_consolidated_map_weighted_mean',
    'deepCoadd_dcr_dra_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e1_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e2_consolidated_map_weighted_mean',
    'deepCoadd_epoch_consolidated_map_max',
    'deepCoadd_epoch_consolidated_map_mean',
    'deepCoadd_epoch_consolidated_map_min',
    'deepCoadd_exposure_time_consolidated_map_sum',
    'deepCoadd_psf_e1_consolidated_map_weighted_mean',
    'deepCoadd_psf_e2_consolidated_map_weighted_mean',
    'deepCoadd_psf_maglim_consolidated_map_weighted_mean',
    'deepCoadd_psf_size_consolidated_map_weighted_mean',
    'deepCoadd_sky_background_consolidated_map_weighted_mean',
    'deepCoadd_sky_noise_consolidated_map_weighted_mean',
]

PLOT_DATASET_TYPES = [
    'propertyMapSurvey_deepCoadd_dcr_ddec_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_dra_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_e1_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_e2_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_max_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_min_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_exposure_time_consolidated_map_sum_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_e1_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_e2_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_maglim_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_size_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_sky_background_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_sky_noise_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
]

print(f'Checking availability in: {COLLECTION}')
rows_hpmap = []
for map_name in HPMAP_DATASET_TYPES:
    short = map_name.replace('deepCoadd_', '').replace('_consolidated', '')
    for band in BANDS:
        ok = False
        try:
            hsp_test = butler.get(
                map_name,
                dataId={'band': band, 'skymap': SKYMAP_NAME},
                collections=COLLECTION
            )
            ok = True
        except Exception:
            pass
        rows_hpmap.append({'dataset_type': map_name, 'short': short, 'band': band, 'ok': ok})

df_hpmap = pd.DataFrame(rows_hpmap)
HPMAPS_AVAILABLE = df_hpmap['ok'].any()
print(f'HealSparse maps available: {HPMAPS_AVAILABLE}')
print(df_hpmap.groupby('dataset_type')['ok'].sum())

In [ ]:
# PNG image availability
rows_plot = []
for plot_name in PLOT_DATASET_TYPES:
    short = (
        plot_name
        .replace('propertyMapSurvey_deepCoadd_', '')
        .replace('_SurveyWidePropertyMapPlot', '')
    )
    for band in BANDS:
        path  = None
        ok    = False
        try:
            uri  = butler.getURI(
                plot_name,
                dataId={'band': band, 'skymap': SKYMAP_NAME},
                collections=COLLECTION
            )
            path = uri.path
            ok   = True
        except Exception:
            pass
        rows_plot.append({
            'dataset_type': plot_name,
            'map': short,
            'band': band,
            'path': path,
            'ok': ok
        })

df = pd.DataFrame(rows_plot)
PLOTS_AVAILABLE   = df['ok'].any()
USE_PLOT_FALLBACK = (not HPMAPS_AVAILABLE) and PLOTS_AVAILABLE
print(f'PNG plots available  : {PLOTS_AVAILABLE}')
print(f'PNG fallback enabled : {USE_PLOT_FALLBACK}')

## 9A. Display configuration flags

- **`SHOW_HPMAP_GRID`**: Display a 2×3 HEALPix map grid (healpy) for each map type.
- **`SHOW_PLOTS_IF_NO_HPMAP`**: Fall back to PNG images if hpmaps are absent.
- **`USE_LOG_NORM`**: Apply a log scale (useful when DDFs dominate the dynamic range).
- **`LOG_LINTHRESH`**: Linear threshold for SymLogNorm.
- **`NSIDE_DISPLAY`**: HEALPix resolution (default: 64).

In [ ]:
# ── Display flags ─────────────────────────────────────────────────────────────
SHOW_HPMAP_GRID        = True
SHOW_PLOTS_IF_NO_HPMAP = True

MAP_TO_DISPLAY  = 'deepCoadd_psf_maglim_consolidated_map_weighted_mean'
PLOT_TO_DISPLAY = 'propertyMapSurvey_deepCoadd_psf_maglim_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot'

# Logarithmic colour scale
USE_LOG_NORM  = True
LOG_LINTHRESH = 1.0

# HEALPix resolution for display
NSIDE_DISPLAY = 32   # set to 128 for higher resolution (slower)

print(f'NSIDE_DISPLAY    = {NSIDE_DISPLAY}')
print(f'USE_LOG_NORM     = {USE_LOG_NORM}')
print(f'HPMAPS_AVAILABLE = {HPMAPS_AVAILABLE}')
print(f'PLOTS_AVAILABLE  = {PLOTS_AVAILABLE}')

## 9B. Helper function — Galactic plane

(already defined in §3C — verification here)

In [ ]:
print(f'GP_RA  : {len(GP_RA)} points, RA ∈ [{GP_RA.min():.1f}, {GP_RA.max():.1f}]°')
print(f'GP_DEC : {len(GP_DEC)} points, Dec ∈ [{GP_DEC.min():.1f}, {GP_DEC.max():.1f}]°')

## 9C. HEALPix display — 2×3 band grid (healpy)

### `read_healsparse.ipynb` approach applied to all available maps

For each available map type:
1. Retrieve the `HealSparseMap` via the Butler,
2. Convert to a dense HEALPix map (`hsp.generate_healpix_map`),
3. Display with `hp.mollview` (Mollweide projection) + graticule + Galactic plane.

The normalisation is computed from the 1st–99th percentile of the first available band,
then applied uniformly to all 6 bands to allow inter-band comparison.

#### 9C-1. Example with a single map and a single band (psf_maglim map, r band)

In [ ]:
if HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    try:
        hsp_demo = butler.get(
            MAP_TO_DISPLAY,
            dataId={'band': BAND_REF, 'skymap': SKYMAP_NAME},
            collections=COLLECTION
        )
        print(f'Type          : {type(hsp_demo)}')
        print(f'Valid pixels  : {hsp_demo.n_valid}')
        print(f'nside_sparse  : {hsp_demo.nside_sparse}')
        print(f'nside_coverage: {hsp_demo.nside_coverage}')

        show_hpmap_healpy(
            hsp_demo,
            title=f'{MAP_TO_DISPLAY}\nband={BAND_REF}  (nside={NSIDE_DISPLAY})',
            unit='mag AB',
            nside=NSIDE_DISPLAY,
            cmap='viridis',
            norm_mode='hist',
            show_gp=True,
        )
    except Exception as e:
        print(f'Error: {type(e).__name__}: {e}')
else:
    print('HealSparse maps not available — section skipped.')

#### 9C-2. 2×3 band grid for the reference map (psf_maglim)

In [ ]:
if SHOW_HPMAP_GRID and HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    print(f'Displaying 6-band grid for: {MAP_TO_DISPLAY}')
    display_hpmap_6bands_healpy(
        MAP_TO_DISPLAY,
        butler,
        nside=NSIDE_DISPLAY,
        use_log=USE_LOG_NORM,
        linthresh=LOG_LINTHRESH,
        cmap='viridis',
    )
else:
    print('HEALPix display skipped (SHOW_HPMAP_GRID=False or no maps available).')

#### 9C-3. Loop over all available map types

In [ ]:
if SHOW_HPMAP_GRID and HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    available_hpmap_names = (
        df_hpmap[df_hpmap['ok']]['dataset_type'].unique().tolist()
    )
    print(f'Displaying {len(available_hpmap_names)} map types...')
    for map_name in available_hpmap_names:
        print(f'\n--- {map_name} ---')
        display_hpmap_6bands_healpy(
            map_name,
            butler,
            nside=NSIDE_DISPLAY,
            use_log=USE_LOG_NORM,
            linthresh=LOG_LINTHRESH,
            cmap='viridis',
        )
else:
    print('healpy loop skipped.')

## 9D. PNG fallback — 2×3 band grid

Triggered only when `USE_PLOT_FALLBACK = True`
(no HealSparseMap available but PNG images exist).

In [ ]:
if USE_PLOT_FALLBACK:
    for plot_name in PLOT_DATASET_TYPES:
        df_check = df[(df['dataset_type'] == plot_name) & df['ok']]
        if df_check.empty:
            continue
        short = (
            plot_name
            .replace('propertyMapSurvey_deepCoadd_', '')
            .replace('_SurveyWidePropertyMapPlot', '')
        )
        print(f'\n--- {short} ---')
        display_plots_6bands(plot_name, df)
else:
    print('PNG fallback skipped (HealSparse maps available or fallback disabled).')

## 10. Map value at the reference field

In [ ]:
if HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    try:
        hspmap_ref = butler.get(
            MAP_TO_DISPLAY,
            dataId={'band': BAND_REF, 'skymap': SKYMAP_NAME},
            collections=COLLECTION
        )
        val_center = hspmap_ref.get_values_pos(RA_CEN, DEC_CEN)
        print(f"Map '{MAP_TO_DISPLAY}' (band {BAND_REF}) at the centre of {FIELD}")
        print(f'  RA={RA_CEN}, Dec={DEC_CEN}  ->  {val_center}')
        val_north = hspmap_ref.get_values_pos(180.0, +60.0)
        print(f'  Sentinel value (outside coverage, RA=180, Dec=+60) -> {val_north}')
    except Exception as e:
        print(f'Error: {type(e).__name__}: {e}')
else:
    print('Map not available — section skipped.')

## 11. Statistics over a region around the reference field

In [ ]:
if HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    try:
        span_dec = 2.0
        span_ra  = span_dec / np.cos(np.deg2rad(DEC_CEN))

        ra  = np.linspace(RA_CEN  - span_ra,  RA_CEN  + span_ra,  250)
        dec = np.linspace(DEC_CEN - span_dec, DEC_CEN + span_dec, 250)
        x, y = np.meshgrid(ra, dec)

        values   = hspmap_ref.get_values_pos(x, y)
        sentinel = hspmap_ref.get_values_pos(180.0, +60.0)
        values   = values.astype(float)
        values[values == sentinel] = np.nan

        print(f'Region {FIELD} — {MAP_TO_DISPLAY} (band {BAND_REF})')
        print(f'  min    = {np.nanmin(values):.4f}')
        print(f'  max    = {np.nanmax(values):.4f}')
        print(f'  mean   = {np.nanmean(values):.4f}')
        print(f'  median = {np.nanmedian(values):.4f}')
        print(f'  std    = {np.nanstd(values):.4f}')
    except Exception as e:
        print(f'Error: {type(e).__name__}: {e}')
else:
    print('Map not available — section skipped.')

## 12. Local visualisation (pcolormesh)

In [ ]:
if HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    try:
        fig, ax = plt.subplots(figsize=(7, 6))
        pcm = ax.pcolormesh(x, y, values, cmap='viridis')
        fig.colorbar(pcm, ax=ax, label=f"{MAP_TO_DISPLAY.split('_')[-2]} ({BAND_REF}-band)")
        ax.set_xlabel('Right Ascension (deg)')
        ax.set_ylabel('Declination (deg)')
        ax.set_title(f'{MAP_TO_DISPLAY}\n{BAND_REF}-band — {FIELD}', fontsize=11)
        ax.invert_xaxis()
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f'Error: {type(e).__name__}: {e}')
else:
    print(f'Map not available for {FIELD} — visualisation skipped.')

## 13. All-sky healpy view — loop over all available maps

For each available map and each band, an individual Mollweide view is produced
with `hp.mollview` (direct approach from `read_healsparse.ipynb`).
The Galactic plane is added as an overlay.

### 13A. Individual healpy view for a single map type, all bands

In [ ]:
if HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    map_name_single = MAP_TO_DISPLAY
    short_single    = map_name_single.replace('deepCoadd_', '').replace('_consolidated', '')

    for band in BANDS:
        try:
            hsp = butler.get(
                map_name_single,
                dataId={'band': band, 'skymap': SKYMAP_NAME},
                collections=COLLECTION
            )
            hpmap = hspmap_to_healpix(hsp, nside=NSIDE_DISPLAY)

            plt.figure(figsize=(12, 6))
            hp.mollview(
                hpmap,
                title=f'{short_single}  —  band = {band}  (nside={NSIDE_DISPLAY})',
                cmap='viridis',
                unit='mag AB',
                norm='hist',
                coord=['C'],
                badcolor='lightgrey',   # colour of hp.UNSEEN / NaN pixels
                bgcolor='white',        # background colour outside the ellipse
            )
            hp.graticule(dpar=30, dmer=30, color='white', alpha=0.5, lw=0.5)
            # Latitude labels
            for lat in [-60, -30, 0, 30, 60]:
                hp.projtext(2, lat, f'{lat:+d}°', lonlat=True, coord='C',
                            color='black', fontsize=8)
            # Longitude labels
            for lon in range(0, 360, 60):
                hp.projtext(lon, 0, f'{lon}°', lonlat=True, coord='C',
                            color='black', fontsize=8)
            # Galactic plane
            overlay_galactic_plane(color='red', lw=1.2, alpha=0.8)
            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f'  [{band}] not available: {type(e).__name__}')
else:
    print('HealSparse maps not available — section skipped.')

### 13B. Loop over all available HEALPix map types — individual views

In [ ]:
if HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    available_types = df_hpmap[df_hpmap['ok']]['dataset_type'].unique().tolist()
    print(f'{len(available_types)} map types to display.')

    for map_name in available_types:
        short  = map_name.replace('deepCoadd_', '').replace('_consolidated', '')
        is_sym = map_name in SYMLOG_MAPS
        print(f'\n=== {short} ===')

        available_bands = (
            df_hpmap[(df_hpmap['dataset_type'] == map_name) & df_hpmap['ok']]
            ['band'].tolist()
        )

        for band in available_bands:
            try:
                hsp   = butler.get(
                    map_name,
                    dataId={'band': band, 'skymap': SKYMAP_NAME},
                    collections=COLLECTION
                )
                hpmap = hspmap_to_healpix(hsp, nside=NSIDE_DISPLAY)
                vmin, vmax = compute_vmin_vmax(hpmap)

                norm_kw = dict(norm='hist') if not is_sym else dict(min=vmin, max=vmax)

                plt.figure(figsize=(12, 6))
                hp.mollview(
                    hpmap,
                    title=f'{short}  —  band = {band}',
                    cmap='viridis',
                    coord=['C'],
                    badcolor='lightgrey',   # colour of hp.UNSEEN / NaN pixels
                    bgcolor='white',        # background colour outside the ellipse
                    **norm_kw
                )
                hp.graticule(dpar=30, dmer=30, color='white', alpha=0.5, lw=0.5)
                for lat in [-60, -30, 0, 30, 60]:
                    hp.projtext(2, lat, f'{lat:+d}°', lonlat=True, coord='C',
                                color='black', fontsize=8)
                for lon in range(0, 360, 60):
                    hp.projtext(lon, 0, f'{lon}°', lonlat=True, coord='C',
                                color='black', fontsize=8)
                overlay_galactic_plane(color='red', lw=1.2, alpha=0.8)
                plt.tight_layout()

                # Optional save to disk
                fig_path = os.path.join(
                    DIR_FIGS,
                    f'healpix_{short}_{band}_nside{NSIDE_DISPLAY}.png'
                )
                plt.savefig(fig_path, dpi=120, bbox_inches='tight')
                plt.show()

            except Exception as e:
                print(f'  [{band}] error: {type(e).__name__}: {e}')
else:
    print('Full map loop skipped.')

## 14. Notes and next steps

### Differences from the reference notebook

| Aspect | `2026-03-11` (reference) | `2026-03-12` (this notebook) |
|---|---|---|
| All-sky projection | `skyproj.McBrydeSkyproj` | `hp.mollview` (native healpy) |
| HealSparse conversion | `skyproj.draw_hspmap` | `hsp.generate_healpix_map` |
| Graticule | `skyproj` built-in | `hp.graticule` + `hp.projtext` |
| Galactic plane | `skyproj` overlay | `hp.projplot` via `overlay_galactic_plane` |
| Normalisation | percentile on `get_values_flat` | percentile on `hpmap[hpmap != hp.UNSEEN]` |

### Next steps

- Use `hp.gnomview` instead of `hp.mollview` for a high-resolution local view.
- Combine `hp.projplot` with object catalogues (sources, clusters, …).
- Compare pixel-by-pixel values across bands via `hp.ud_grade`.
- Export HEALPix maps to FITS format with `hp.write_map`.
- Extend the DP1 vs DP2 comparison over the same nside and fields.